# GPT-2 text generation

Comparing decoding strategies on GPT-2: greedy, top-k, and nucleus (top-p). I use the built-in `generate` first, then write top-k and top-p by hand to see what they actually do to the logits.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()

PROMPT = "The strangest thing about learning to code is"

def generate(prompt, **kwargs):
    ids = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(**ids, max_new_tokens=60, pad_token_id=tok.eos_token_id, **kwargs)
    return tok.decode(out[0], skip_special_tokens=True)

## Greedy

Always picks the most likely token. Deterministic, and it tends to loop.

In [ ]:
print(generate(PROMPT, do_sample=False))

## Top-k and nucleus sampling

Run each one a few times with different seeds to see how much the output changes.

In [ ]:
for seed in range(3):
    torch.manual_seed(seed)
    print(f"--- top-k=50, seed {seed}")
    print(generate(PROMPT, do_sample=True, top_k=50))

In [ ]:
for seed in range(3):
    torch.manual_seed(seed)
    print(f"--- top-p=0.9, seed {seed}")
    print(generate(PROMPT, do_sample=True, top_p=0.9, top_k=0))

## Writing the sampling by hand

`top_k` keeps the k highest logits. `top_p` keeps the smallest set of tokens whose probabilities add up to p. Everything else gets set to -inf before the softmax.

In [ ]:
def sample_next(logits, top_k=0, top_p=1.0, temperature=1.0):
    logits = logits / temperature
    if top_k > 0:
        kth = torch.topk(logits, top_k).values[-1]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        cum = probs.cumsum(-1)
        # drop a token if the tokens before it already cover top_p
        remove = (cum - probs) > top_p
        sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(0, sorted_idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1)


@torch.no_grad()
def generate_manual(prompt, n_tokens=60, **sampling):
    ids = tok(prompt, return_tensors="pt").input_ids.to(device)
    for _ in range(n_tokens):
        logits = model(ids).logits[0, -1]
        nxt = sample_next(logits, **sampling)
        ids = torch.cat([ids, nxt.view(1, 1)], dim=1)
    return tok.decode(ids[0])

torch.manual_seed(0)
print(generate_manual(PROMPT, top_p=0.9))

## How many tokens survive the cutoff?

Tracking how many tokens are left after top-p at each step. When the model is sure, it's one or two. When it isn't, it can be hundreds.

In [ ]:
@torch.no_grad()
def nucleus_sizes(prompt, n_tokens=30, top_p=0.9):
    ids = tok(prompt, return_tensors="pt").input_ids.to(device)
    sizes = []
    for _ in range(n_tokens):
        logits = model(ids).logits[0, -1]
        probs = logits.softmax(-1)
        sorted_probs, _ = torch.sort(probs, descending=True)
        cum = sorted_probs.cumsum(-1)
        sizes.append(int(((cum - sorted_probs) <= top_p).sum()))
        ids = torch.cat([ids, torch.multinomial(probs, 1).view(1, 1)], dim=1)
    return sizes

sizes = nucleus_sizes(PROMPT)
print(sizes)
print("median:", sorted(sizes)[len(sizes) // 2])